In [186]:

import cv2
import time
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

In [187]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="../Mini-Project-2/face_landmarker.task"),
    running_mode=VisionRunningMode.VIDEO,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

In [188]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_ear(eye_points):
    A = euclidean(eye_points[1], eye_points[5])
    B = euclidean(eye_points[2], eye_points[4])
    C = euclidean(eye_points[0], eye_points[3])

    ear = (A + B) / (2.0 * C)
    return ear

In [189]:
EAR_THRESHOLD = 0.20
BLINK_FRAMES = 3

blink = False
blink_display_frames = 15   # show text for 20 frames
blink_display_counter = 0

HEAD_THRESHOLD = 15
head_movement_display_frames = 15
head_display_counter = 0
head_direction = ""

In [190]:
# Head pose landmarks
NOSE_TIP = 4
FOREHEAD = 10
CHIN     = 152
L_CHEEK  = 234
R_CHEEK  = 454

# Iris landmarks
L_IRIS      = 468
L_EYE_LEFT  = 33
L_EYE_RIGHT = 133
R_IRIS      = 473
R_EYE_LEFT  = 362
R_EYE_RIGHT = 263

# Blend weights for yaw
HEAD_WEIGHT = 0
IRIS_WEIGHT = 1

# Smoothing
SMOOTH_WINDOW = 10
yaw_history   = []
ear_history = []
pitch_history = []

left_eye_indices = [33, 160, 158, 133, 153, 144]
right_eye_indices = [362, 385, 387, 263, 373, 380]
print('Indices and constants set!')

Indices and constants set!


In [191]:
# def get_head_yaw(landmarks):
#     """
#     Returns horizontal head turn as a normalized float.
#     Negative = left, Positive = right.
#     Typical range: -0.4 to 0.4
#     """
#     nose    = landmarks[NOSE_TIP]
#     l_cheek = landmarks[L_CHEEK]
#     r_cheek = landmarks[R_CHEEK]

#     face_center_x = (l_cheek.x + r_cheek.x) / 2
#     face_width    =  r_cheek.x - l_cheek.x

#     if face_width < 1e-6:
#         return None

#     return (nose.x - face_center_x) / face_width


# def get_head_pitch(landmarks):
#     """
#     Returns vertical head tilt as a ratio.
#     Higher value = looking up, Lower = looking down.
#     Typical range: 0.9 (down) to 1.5 (up)
#     """
#     nose = landmarks[NOSE_TIP]
#     fore = landmarks[FOREHEAD]
#     chin = landmarks[CHIN]

#     nose_to_chin     = abs(nose.y - chin.y)
#     forehead_to_nose = abs(fore.y - nose.y)

#     if forehead_to_nose < 1e-6:
#         return None

#     return nose_to_chin / forehead_to_nose

def get_iris_x_offset(landmarks):
    """
    Returns average iris X offset across both eyes.
    0.0 = center, negative = left, positive = right.
    Typical range: -0.15 to 0.15
    """
    def eye_offset(iris_idx, left_idx, right_idx):
        iris  = landmarks[iris_idx]
        left  = landmarks[left_idx]
        right = landmarks[right_idx]
        eye_w = abs(right.x - left.x)
        if eye_w < 1e-6:
            return None
        return ((iris.x - left.x) / eye_w) - 0.5

    l = eye_offset(L_IRIS, L_EYE_LEFT, L_EYE_RIGHT)
    r = eye_offset(R_IRIS, R_EYE_LEFT, R_EYE_RIGHT)

    if l is None and r is None: return None
    if l is None: return r
    if r is None: return l
    return (l + r) / 2.0


def smooth(value, history):
    """
    Adds value to history buffer and returns smoothed average.
    Reduces jitter from frame-to-frame noise.
    """
    history.append(value)
    if len(history) > SMOOTH_WINDOW:
        history.pop(0)
    return sum(history) / len(history)

In [192]:
WHITE  = (255, 255, 255)
GREEN  = (0, 255, 0)
YELLOW = (0, 255, 255)
RED    = (0, 0, 255)
CYAN   = (255, 255, 0)

In [193]:
def calc_ear(frame, landmarks, h, w):
    
    # nose landmarks
    # nose = landmarks[1]  # nose tip
    # nose_x = int(nose.x * w)

    # frame_center_x = w // 2

    # offset = nose_x - frame_center_x

    # if offset > HEAD_THRESHOLD:
    #     head_direction = "Looking Right"
    #     head_display_counter = head_movement_display_frames
    # elif offset < -HEAD_THRESHOLD:
    #     head_direction = "Looking Left"
    #     head_display_counter = head_movement_display_frames
    # else:
    #     head_direction = "Facing Center"
        
        
    # LEFT EYE
    left_eye_points = []

    for idx in left_eye_indices:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        left_eye_points.append((x, y))
        # cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

    # RIGHT EYE
    right_eye_points = []

    for idx in right_eye_indices:
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        right_eye_points.append((x, y))
        # cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

    # Compute EAR
    ear_left = calculate_ear(left_eye_points)
    ear_right = calculate_ear(right_eye_points)

    ear = (ear_left + ear_right) / 2
    
    # blink logic
    # if ear < EAR_THRESHOLD:
    #     blink_counter += 1
    # else:
    #     if blink_counter >= BLINK_FRAMES:
    #         blink_display_counter = blink_display_frames
    #     blink_counter = 0
        
    # if head_display_counter > 0:
    #     if head_direction == "Facing Center":
    #         color = (0, 255, 0)
    #     else:
    #         color = (0, 0, 255)
        
    #     cv2.putText(frame, head_direction, (20, 120),
    #                     cv2.FONT_HERSHEY_SIMPLEX, 1,
    #                     color, 2)
    #     head_display_counter -= 1
    
    # display EAR value
    cv2.putText(frame, f"EAR: {ear:.2f}", (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX, 1,
        (255, 0, 0), 2)   
    
    return ear

    # if blink_display_counter > 0:
    #     cv2.putText(frame, "Blink Detected", (20, 80),
    #                 cv2.FONT_HERSHEY_SIMPLEX, 1,
    #                 (0, 255, 0), 2)
    #     blink_display_counter -= 1
    
# else:
#     blink_counter = 0

In [194]:
def landmark(frame, lm, h, w, ear):
        
    nose_tip = lm[NOSE_TIP]
    forehead = lm[FOREHEAD]
    chin = lm[CHIN]
    l_cheek = lm[L_CHEEK]
    r_cheek = lm[R_CHEEK]

    # ── Compute yaw, pitch, iris ─────────────────────────────────
    # head_yaw  = get_head_yaw(lm)
    # head_pitch = get_head_pitch(lm)
    iris_x    = get_iris_x_offset(lm)

    # Blend yaw: head + iris
    # if head_yaw is not None and iris_x is not None:
    #     raw_yaw = head_yaw * HEAD_WEIGHT + iris_x * IRIS_WEIGHT
    # else:
    #     raw_yaw = head_yaw

    # Smooth
    yaw   = smooth(iris_x,    yaw_history)   if iris_x    is not None else None
    ear   = smooth(ear,    ear_history)   if ear    is not None else None
    # yaw   = smooth(raw_yaw,    yaw_history)   if raw_yaw    is not None else None
    # pitch = smooth(head_pitch, pitch_history)  if head_pitch is not None else None

    # ── Draw landmark reference dots ─────────────────────────────
    for idx, color in [
        (NOSE_TIP, GREEN),
        (FOREHEAD, CYAN),
        (CHIN,     CYAN),
        (L_CHEEK,  RED),
        (R_CHEEK,  RED),
    ]:
        px = int(lm[idx].x * w)
        py = int(lm[idx].y * h)
        cv2.circle(frame, (px, py), 5, color, -1)
        
    # # face lines
    # cv2.line(frame, (int(forehead.x*w), int(forehead.y*h)), (int(nose_tip.x*w), int(nose_tip.y*h)), GREEN, 1)
    # cv2.line(frame, (int(nose_tip.x*w), int(nose_tip.y*h)), (int(chin.x*w), int(chin.y*h)), GREEN, 1)
    # cv2.line(frame, (int(l_cheek.x*w), int(l_cheek.y*h)), (int(nose_tip.x*w), int(nose_tip.y*h)), GREEN, 1)
    # cv2.line(frame, (int(nose_tip.x*w), int(nose_tip.y*h)), (int(r_cheek.x*w), int(r_cheek.y*h)), GREEN, 1)
    
    

    # ── Draw iris dots ───────────────────────────────────────────
    for iris_idx in [L_IRIS, R_IRIS]:
        px = int(lm[iris_idx].x * w)
        py = int(lm[iris_idx].y * h)
        cv2.circle(frame, (px, py), 4, YELLOW, -1)
    
        
    # ── Display values ───────────────────────────────────────────
    lines = [
    #     (f'Head Yaw:   {head_yaw:.3f}',  WHITE),
        (f'EAR (Y):     {ear:.3f}',    YELLOW),
        (f'Iris (X):     {iris_x:.3f}',    YELLOW),
        # (f'Blended Yaw:{yaw:.3f}',        GREEN),
    #     (f'Pitch:      {pitch:.3f}',      CYAN),
    ] if (yaw and ear) else []

    for i, (text, color) in enumerate(lines):
        cv2.putText(frame, text, (20, h - 20 - i * 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 1)
        
    return yaw, ear

In [195]:
def calc_y(frame, ear, h, w):
    pass

In [196]:
X_SENSITIVITY = 5.0
Y_SENSITIVITY = 20.0
EAR_CENTER = 0.26

def draw_crosshair(frame, yaw, ear, h, w):
    if yaw is not None and ear is not None:
        # Map to screen — raw estimate (no calibration in this notebook)
        cx = int(w * 0.5 + yaw   * w * X_SENSITIVITY)
        cy = int(h * 0.5 - (ear - EAR_CENTER) * h * Y_SENSITIVITY)
        cx = max(20, min(w - 20, cx))
        cy = max(20, min(h - 20, cy))

        cv2.circle(frame, (cx, cy), 18, YELLOW, 2)
        cv2.circle(frame, (cx, cy), 4,  YELLOW, -1)
        cv2.line(frame, (cx - 25, cy), (cx + 25, cy), YELLOW, 1)
        cv2.line(frame, (cx, cy - 25), (cx, cy + 25), YELLOW, 1)

In [197]:
def crop_to_fill(frame, target_w, target_h):
    """
    Crops frame to fill target dimensions while preserving aspect ratio.
    Centers the crop — no stretching, no black bars.
    """

    src_h, src_w = frame.shape[:2]

    # Scale factor to fill target — use the larger ratio
    scale = max(target_w / src_w, target_h / src_h)

    # Scaled dimensions
    scaled_w = int(src_w * scale)
    scaled_h = int(src_h * scale)

    # Resize
    resized = cv2.resize(frame, (scaled_w, scaled_h))

    # Center crop
    x1 = (scaled_w - target_w) // 2
    y1 = (scaled_h - target_h) // 2

    return resized[y1:y1 + target_h, x1:x1 + target_w]

In [198]:
cap   = cv2.VideoCapture(0)
ret, frame = cap.read()

# Create fullscreen window first to get screen dimensions
cv2.namedWindow("Frame", cv2.WINDOW_NORMAL)
cv2.setWindowProperty("Frame", cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

# Get screen size from a temporary fullscreen display
cv2.imshow("Frame", frame)
cv2.waitKey(1)
screen_w = cv2.getWindowImageRect("Frame")[2]
screen_h = cv2.getWindowImageRect("Frame")[3]

In [199]:
cap = cv2.VideoCapture(0)
yaw_history   = []
ear_history = []

while True:
    
    ret, frame = cap.read()
    
    if not ret:
        break
    
    h, w  = frame.shape[:2]
    
    frame = cv2.flip(frame, 1)
    frame = crop_to_fill(frame, screen_w, screen_h)
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result    = landmarker.detect_for_video(mp_image, int(time.time() * 1000))
    
    
    if result.face_landmarks:
        lm = result.face_landmarks[0]
        ear = calc_ear(frame, lm, h, w)
        yaw, ear = landmark(frame, lm, h, w, ear)
        draw_crosshair(frame, yaw, ear, h, w)
    else:
        cv2.putText(frame, 'No face detected', (20, h - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, RED, 1)
    
    cv2.imshow("Frame", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv2.destroyAllWindows()